# Week 3 - SQL Queries: Program Performance
### Name: Syed Muhammed Ahmed

**Objective:** Practice SQL querying across related tables (`applicants`, `interns`, `hackathon_scores`) stored in `nextgen.db`.

**Note on the data:** `nextgen.db` here is a mock database I generated myself, with realistic foreign-key relationships between the three tables: `applicant_id` links `applicants` -> `interns`, and `intern_id` links `interns` -> `hackathon_scores`. Not every applicant becomes an intern, and not every intern reaches the hackathon, which is what makes Query 4's LEFT JOIN meaningful.

In [1]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("nextgen.db")

## Step 2: Understand the 3 Tables and How They Relate

Before writing any joins, look at each table individually.

In [2]:
pd.read_sql("SELECT * FROM applicants LIMIT 5;", conn)

,applicant_id,name,domain,university,application_date,status
0,1,Matthew Jones,Mobile App Development,IBA Karachi,2025-12-03,Under Review
1,2,Anthony Romero,Web Development,Bahria University,2026-01-11,Selected
2,3,Benjamin Davis,Web Development,NUST,2025-10-11,Selected
3,4,Becky Walker,Cybersecurity,FAST-NUCES,2025-12-21,Selected
4,5,Laura Sanders,UI/UX Design,COMSATS,2026-01-27,Rejected


In [3]:
pd.read_sql("SELECT * FROM interns LIMIT 5;", conn)

,intern_id,applicant_id,domain,start_date,completion_status
0,1,2,Web Development,2026-01-23,Completed
1,2,3,Web Development,2025-10-27,Completed
2,3,4,Cybersecurity,2026-01-02,Completed
3,4,7,Cybersecurity,2025-11-22,Completed
4,5,11,UI/UX Design,2026-01-14,Completed


In [4]:
pd.read_sql("SELECT * FROM hackathon_scores LIMIT 5;", conn)

,intern_id,score,domain
0,1,83,Web Development
1,2,85,Web Development
2,3,83,Cybersecurity
3,4,67,Cybersecurity
4,5,86,UI/UX Design


In [5]:
# Row counts for context - not every applicant becomes an intern,
# and not every intern reaches the hackathon
for table in ["applicants", "interns", "hackathon_scores"]:
    n = pd.read_sql(f"SELECT COUNT(*) AS n FROM {table};", conn).iloc[0]["n"]
    print(table, "->", n, "rows")

applicants -> 160 rows
interns -> 61 rows
hackathon_scores -> 48 rows


**Table relationships confirmed:**
- `applicants.applicant_id` -> `interns.applicant_id` (only `Selected` applicants who actually started the program appear in `interns`)
- `interns.intern_id` -> `hackathon_scores.intern_id` (only interns who reached the end-of-program hackathon have a score row — most `Completed` interns do, but a couple of `Dropped Out` interns also have a score because they left *after* the hackathon)

Every JOIN below relies on matching these ID columns.

## Step 3: Part A - 5 SQL Queries

### Query 1 - How many interns completed each domain's program?

In [6]:
# Counts completed interns grouped by domain
query1 = """
SELECT domain, COUNT(*) AS completed_count
FROM interns
WHERE completion_status = 'Completed'
GROUP BY domain
ORDER BY completed_count DESC;
"""
pd.read_sql(query1, conn)

,domain,completed_count
0,Mobile App Development,10
1,Data Science,10
2,UI/UX Design,9
3,Cybersecurity,9
4,Web Development,8


`GROUP BY domain` bundles all rows with the same domain together, and `COUNT(*)` counts how many rows fall in each bundle. The `WHERE` clause filters to only completed interns before grouping, so dropouts don't inflate the count, and `ORDER BY ... DESC` puts the most successful domain first.

### Query 2 - What is the average hackathon score per domain?

In [7]:
# Calculates the average hackathon score for each domain
query2 = """
SELECT domain, ROUND(AVG(score), 2) AS avg_score
FROM hackathon_scores
GROUP BY domain
ORDER BY avg_score DESC;
"""
pd.read_sql(query2, conn)

,domain,avg_score
0,Web Development,81.50
1,Mobile App Development,80.80
2,UI/UX Design,79.89
3,Cybersecurity,75.60
4,Data Science,75.55


`AVG(score)` calculates the mean score within each domain group, and `ROUND(..., 2)` limits the result to 2 decimal places so it's actually readable instead of a long float.

### Query 3 - Which interns scored above a threshold (top performers)?

In [8]:
# Lists interns who scored 85 or above, as candidates for showcase/certificates
query3 = """
SELECT i.intern_id, i.domain, h.score
FROM interns i
JOIN hackathon_scores h ON i.intern_id = h.intern_id
WHERE h.score >= 85
ORDER BY h.score DESC;
"""
pd.read_sql(query3, conn)

,intern_id,domain,score
0,7,Cybersecurity,95
1,60,Mobile App Development,92
2,39,Mobile App Development,91
3,16,Data Science,90
4,17,Data Science,90
5,45,Mobile App Development,90
6,24,UI/UX Design,89
7,46,Web Development,88
8,48,Web Development,87
9,5,UI/UX Design,86


This is the first `JOIN`: intern details live in `interns`, but scores live in `hackathon_scores`, so a `JOIN ... ON i.intern_id = h.intern_id` matches rows from both tables wherever the `intern_id` values agree. `WHERE h.score >= 85` filters to top performers. Table aliases `i` and `h` keep the query readable instead of repeating full table names.

### Query 4 - Conversion rate from "applied" to "completed" per domain

In [9]:
# Compares total applicants vs completed interns per domain to find conversion rate
query4 = """
SELECT
    a.domain,
    COUNT(DISTINCT a.applicant_id) AS total_applicants,
    COUNT(DISTINCT i.intern_id) AS total_completed,
    ROUND(
        100.0 * COUNT(DISTINCT i.intern_id) / COUNT(DISTINCT a.applicant_id),
        2
    ) AS conversion_rate_pct
FROM applicants a
LEFT JOIN interns i
    ON a.applicant_id = i.applicant_id AND i.completion_status = 'Completed'
GROUP BY a.domain
ORDER BY conversion_rate_pct DESC;
"""
pd.read_sql(query4, conn)

,domain,total_applicants,total_completed,conversion_rate_pct
0,Mobile App Development,29,10,34.48
1,Data Science,30,10,33.33
2,Web Development,29,8,27.59
3,Cybersecurity,33,9,27.27
4,UI/UX Design,39,9,23.08


This is the first `LEFT JOIN`: it keeps every row from `applicants`, even applicants who were never selected or never completed the program. A regular `JOIN` would only keep applicants who *did* become completed interns, which would make the conversion rate meaningless (it would show 100% every time, since everyone who didn't convert would already be excluded). `COUNT(DISTINCT ...)` avoids double-counting, and `100.0 *` (instead of `100 *`) forces decimal math so the percentage isn't incorrectly rounded down to a whole number.

### Query 5 - Score distribution across ranges (my choice)

In [10]:
# Buckets all hackathon scores into 10-point ranges to see the overall score distribution
query5 = """
SELECT
    CASE
        WHEN score >= 90 THEN '90-100'
        WHEN score >= 80 THEN '80-89'
        WHEN score >= 70 THEN '70-79'
        WHEN score >= 60 THEN '60-69'
        ELSE 'Below 60'
    END AS score_range,
    COUNT(*) AS num_interns
FROM hackathon_scores
GROUP BY score_range
ORDER BY score_range DESC;
"""
pd.read_sql(query5, conn)

,score_range,num_interns
0,Below 60,1
1,90-100,6
2,80-89,20
3,70-79,11
4,60-69,10


I picked this because it's more useful for a program manager than a single average score — it shows the *shape* of performance (e.g. is it a tight cluster around 75, or a wide spread with a lot of strong and a lot of weak performers?). It uses a `CASE` expression plus `GROUP BY` and `COUNT(*)`, satisfying the "at least one JOIN/GROUP BY/aggregate" requirement.

## Wrap-up

These 5 queries (plus the exploration in Step 2) cover: simple aggregation with `GROUP BY` (Query 1), `AVG`/`ROUND` (Query 2), an inner `JOIN` (Query 3), a `LEFT JOIN` for a funnel/conversion calculation (Query 4), and a `CASE`-based bucketing query (Query 5). The dashboard in `dashboard.py` reuses this same logic to build an interactive view on top of it.

In [11]:
conn.close()